In [10]:
from pathlib import Path
import json

DATA_DIR = Path("dataset")

print(DATA_DIR.exists())

True


In [12]:
with open("dataset/code_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

In [13]:
with open("dataset/eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)


In [14]:
with open("dataset/categories.json", encoding="utf-8") as f:
    categories = json.load(f)["categories"]

In [15]:
print(len(corpus))      
print(len(questions))   
print(len(categories))

200
25
5


In [16]:
corpus[0]

{'id': 'func_001',
 'language': 'python',
 'function_name': 'verify_jwt_token',
 'category': 'auth',
 'description': 'Проверяет JWT-токен и возвращает payload или причину невалидности.',
 'code': 'def verify_jwt_token(token: str, secret: str) -> dict:\n    """Проверяет JWT-токен и возвращает payload или причину невалидности."""\n    try:\n        payload = jwt.decode(token, secret, algorithms=["HS256"])\n        return {"valid": True, "data": payload}\n    except jwt.ExpiredSignatureError:\n        return {"valid": False, "error": "expired"}\n    except jwt.InvalidTokenError:\n        return {"valid": False, "error": "invalid"}'}

In [17]:
print(corpus[0].keys())
print(questions[0].keys())

dict_keys(['id', 'language', 'function_name', 'category', 'description', 'code'])
dict_keys(['question_id', 'query', 'language', 'correct_chunk_id'])


In [18]:
languages = []

for item in corpus:
    languages.append(item["language"])

set(languages)

{'java', 'python'}

In [19]:
from collections import Counter

Counter(languages)

Counter({'python': 100, 'java': 100})

In [20]:
category_names = []

for item in corpus:
    category_names.append(item["category"])

set(category_names)

{'auth', 'database', 'http', 'utils', 'validation'}

In [21]:
Counter(category_names)

Counter({'auth': 40,
         'database': 40,
         'http': 40,
         'validation': 40,
         'utils': 40})

In [22]:
question_languages = []

for question in questions:
    question_languages.append(question["language"])

Counter(question_languages)

Counter({'ru': 15, 'en': 10})

In [23]:
questions[0]

{'question_id': 'q_01',
 'query': 'как проверить, истёк ли токен?',
 'language': 'ru',
 'correct_chunk_id': 'func_001'}

In [24]:
import pandas as pd

corpus_df = pd.DataFrame(corpus)
questions_df = pd.DataFrame(questions)

In [25]:
corpus_df.head()

,id,language,function_name,category,description,code
0,func_001,python,verify_jwt_token,auth,Проверяет JWT-токен и возвращает payload или п...,"def verify_jwt_token(token: str, secret: str) ..."
1,func_002,python,hash_password,auth,Хэширует пароль с помощью bcrypt с cost-фактор...,def hash_password(password: str) -> str:\n ...
2,func_003,python,check_password,auth,Сравнивает открытый пароль с bcrypt-хэшем и во...,"def check_password(plain: str, hashed: str) ->..."
3,func_004,python,generate_session_id,auth,Генерирует уникальный идентификатор сессии на ...,"def generate_session_id() -> str:\n """"""Гене..."
4,func_005,python,validate_credentials,auth,"Проверяет, что логин и пароль непустые и соотв...","def validate_credentials(username: str, passwo..."


In [26]:
questions_df.head()

,question_id,query,language,correct_chunk_id
0,q_01,"как проверить, истёк ли токен?",ru,func_001
1,q_02,where is password hashing implemented,en,func_002
2,q_03,проверка двухфакторной аутентификации по коду,ru,func_014
3,q_04,logout user and clear session,en,func_010
4,q_05,проверить права администратора на эндпоинте,ru,func_109


In [27]:
corpus_df["language"].value_counts()

language
python    100
java      100
Name: count, dtype: int64

In [28]:
corpus_df["category"].value_counts()

category
auth          40
database      40
http          40
validation    40
utils         40
Name: count, dtype: int64

In [29]:
questions_df["language"].value_counts()

language
ru    15
en    10
Name: count, dtype: int64

### Вывод по датасету

В датасете 200 фрагментов кода и 25 тестовых вопросов. Корпус содержит два языка программирования: Python и Java, по 100 функций каждого языка. Все функции распределены по пяти категориям: auth, database, http, validation и utils, по 40 функций в каждой категории. Вопросы представлены на двух языках: 15 русских и 10 английских.

In [33]:
def build_function_text(row):
    return (
        f"Имя функции: {row['function_name']} "
        f"Описание: {row['description']} "
        f"Код:\n{row['code']}"
    )

In [35]:
corpus_df["text_for_embedding"] = corpus_df.apply(build_function_text, axis=1)

In [36]:
corpus_df[["id", "function_name", "text_for_embedding"]].head()

,id,function_name,text_for_embedding
0,func_001,verify_jwt_token,Имя функции: verify_jwt_token Описание: Провер...
1,func_002,hash_password,Имя функции: hash_password Описание: Хэширует ...
2,func_003,check_password,Имя функции: check_password Описание: Сравнива...
3,func_004,generate_session_id,Имя функции: generate_session_id Описание: Ген...
4,func_005,validate_credentials,Имя функции: validate_credentials Описание: Пр...


In [37]:
corpus_texts = corpus_df["text_for_embedding"].tolist()
corpus_ids = corpus_df["id"].tolist()

In [38]:
print(len(corpus_texts))
print(len(corpus_ids))
print(corpus_ids[:5])

200
200
['func_001', 'func_002', 'func_003', 'func_004', 'func_005']


In [40]:
query_texts = questions_df["query"].tolist()
correct_ids = questions_df["correct_chunk_id"].tolist()

In [41]:
print(len(query_texts))
print(len(correct_ids))
print(query_texts[:3])
print(correct_ids[:3])

25
25
['как проверить, истёк ли токен?', 'where is password hashing implemented', 'проверка двухфакторной аутентификации по коду']
['func_001', 'func_002', 'func_014']
